# singer_v1 — DiffSinger Acoustic Training

Train a DiffSinger acoustic model from scratch on the singer_v1 dataset (94 clips, ~12 min).

**Before first run:** Upload `singer_v1_binary.zip` (27 MB) to your Google Drive root.

**Runtime:** GPU → T4. Training takes ~9h total across multiple sessions. Checkpoints are saved to Google Drive and survive session restarts.

**Every session:** Run All cells. It automatically resumes from the last checkpoint.

## 1. Mount Google Drive & Setup

In [ ]:
# Mount Google Drive (checkpoints survive session restarts here)
from google.colab import drive
drive.mount('/content/drive')

# Create persistent workspace on Drive
import os
DRIVE_DIR = '/content/drive/MyDrive/diffsinger_training'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive workspace: {DRIVE_DIR}')

# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Clone DiffSinger
!git clone https://github.com/openvpi/DiffSinger.git /content/DiffSinger
%cd /content/DiffSinger

In [ ]:
# Install dependencies (PyTorch already in Colab)
!pip install -q lightning~=2.3.0 librosa==0.9.2 scipy>=1.10.0 pyworld==0.3.4 \
    praat-parselmouth==0.4.3 einops>=0.7.0 onnx~=1.16.0 tensorboardX \
    torchmetrics click tqdm PyYAML h5py MonkeyType==23.3.0

## 2. Upload data & download vocoder

In [ ]:
import os

# Use binary zip from Google Drive (upload it there once, reuse every session)
binary_zip = '/content/drive/MyDrive/singer_v1_binary.zip'
if not os.path.exists(binary_zip):
    raise FileNotFoundError(
        'Upload singer_v1_binary.zip to your Google Drive root first!\n'
        'Go to drive.google.com and drop the file there.'
    )

!mkdir -p /content/DiffSinger/data/singer_v1
!unzip -o "$binary_zip" -d /content/DiffSinger/data/singer_v1/
!ls -lh /content/DiffSinger/data/singer_v1/binary/

In [ ]:
# Download OpenVPI vocoder from GitHub releases
import json, urllib.request, zipfile, shutil

VOCODER_DIR = '/content/DiffSinger/checkpoints/pc_nsf_hifigan_44.1k_hop512_128bin_2025.02'
if not os.path.exists(VOCODER_DIR):
    print('Downloading vocoder...')
    api_url = 'https://api.github.com/repos/openvpi/vocoders/releases/tags/pc-nsf-hifigan-44.1k-hop512-128bin-2025.02'
    with urllib.request.urlopen(api_url) as resp:
        release = json.loads(resp.read())
    # Find the zip asset (exclude onnx/openutau)
    asset = [a for a in release['assets']
             if 'onnx' not in a['name'].lower()
             and 'openutau' not in a['name'].lower()
             and 'oudep' not in a['name'].lower()][0]
    zip_path = f"/content/{asset['name']}"
    urllib.request.urlretrieve(asset['browser_download_url'], zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/DiffSinger/checkpoints/')
    os.remove(zip_path)
    print(f'Vocoder ready: {os.listdir(VOCODER_DIR)}')
else:
    print('Vocoder already present.')

## 3. Training config

In [ ]:
config = """
base_config:
  - configs/acoustic.yaml

# Dataset
dictionaries:
  en: dictionaries/english_ipa.txt

datasets:
  - raw_data_dir: data/singer_v1/raw
    speaker: singer_v1
    spk_id: 0
    language: en
    test_prefixes:
      - '000110'
      - '000019'
      - '000004'
      - '000051'
      - '000047'

binary_data_dir: data/singer_v1/binary

# Vocoder
vocoder: NsfHifiGAN
vocoder_ckpt: checkpoints/pc_nsf_hifigan_44.1k_hop512_128bin_2025.02/model.ckpt

# Audio (match vocoder)
audio_sample_rate: 44100
audio_num_mel_bins: 128
hop_size: 512
fft_size: 2048
win_size: 2048
fmin: 40
fmax: 16000

# Pitch extraction
pe: rmvpe
pe_ckpt: checkpoints/rmvpe/model.pt

# Harmonic-noise separation
hnsep: world

# Single speaker, single language
use_spk_id: false
num_spk: 1
use_lang_id: false
num_lang: 1

# Backbone — 512 channels fits T4 16GB comfortably
backbone_args:
  num_channels: 512
  num_layers: 6
  kernel_size: 31
  dropout_rate: 0.1
  strong_cond: true

# Disable shallow diffusion to save VRAM
use_shallow_diffusion: false

# Training from scratch
finetune_enabled: false
finetune_ckpt_path: null

# Training
optimizer_args:
  optimizer_cls: torch.optim.AdamW
  lr: 0.0006
  beta1: 0.9
  beta2: 0.98
  weight_decay: 0
lr_scheduler_args:
  scheduler_cls: torch.optim.lr_scheduler.StepLR
  step_size: 5000
  gamma: 0.75

max_batch_frames: 24000
max_batch_size: 8
max_updates: 160000
val_check_interval: 2000
num_valid_plots: 5
num_ckpt_keep: 3
permanent_ckpt_start: 20000
permanent_ckpt_interval: 20000

# GPU training
pl_trainer_precision: 16-mixed

# Binarization (already done — skip this step)
binarization_args:
  shuffle: true
  num_workers: 2

# No augmentation for initial run
augmentation_args:
  random_pitch_shifting:
    enabled: false
  fixed_pitch_shifting:
    enabled: false
  random_time_stretching:
    enabled: false
""".strip()

with open('/content/DiffSinger/configs/singer_v1_acoustic.yaml', 'w') as f:
    f.write(config)

print('Config written.')

In [ ]:
# Write the english_ipa dictionary (not in upstream DiffSinger repo)
import os

dictionary = """\
a\ta
aj\taj
aw\taw
aː\taː
b\tb
bʲ\tbʲ
c\tc
cʰ\tcʰ
cʷ\tcʷ
d\td
dʒ\tdʒ
dʲ\tdʲ
d̪\td̪
e\te
ej\tej
eː\teː
f\tf
fʲ\tfʲ
h\th
i\ti
iː\tiː
j\tj
k\tk
kʰ\tkʰ
l\tl
m\tm
mʲ\tmʲ
m̩\tm̩
n\tn
o\to
oː\toː
p\tp
pʲ\tpʲ
s\ts
t\tt
tʃ\ttʃ
tʰ\ttʰ
tʲ\ttʲ
t̪\tt̪
u\tu
uː\tuː
v\tv
vʲ\tvʲ
w\tw
z\tz
æ\tæ
ç\tç
ð\tð
ŋ\tŋ
ɐ\tɐ
ɑ\tɑ
ɑː\tɑː
ɒ\tɒ
ɒː\tɒː
ɔ\tɔ
ɔj\tɔj
ɖ\tɖ
ə\tə
əw\təw
ɚ\tɚ
ɛ\tɛ
ɛː\tɛː
ɜ\tɜ
ɜː\tɜː
ɟ\tɟ
ɡ\tɡ
ɪ\tɪ
ɫ\tɫ
ɲ\tɲ
ɹ\tɹ
ʃ\tʃ
ʈ\tʈ
ʈʲ\tʈʲ
ʉː\tʉː
ʊ\tʊ
ʋ\tʋ
ʎ\tʎ
θ\tθ
"""

os.makedirs('/content/DiffSinger/dictionaries', exist_ok=True)
with open('/content/DiffSinger/dictionaries/english_ipa.txt', 'w', encoding='utf-8') as f:
    f.write(dictionary)
print('Dictionary written (78 IPA symbols).')

## 4. Train

Checkpoints save to Google Drive. If the session dies, just start a new session and **Run All** — it resumes automatically.

In [ ]:
import os

%env PYTHONPATH=/content/DiffSinger
%env TORCH_CUDNN_V8_API_ENABLED=1

# Symlink checkpoint dir to Google Drive so checkpoints persist
DRIVE_CKPT = '/content/drive/MyDrive/diffsinger_training/singer_v1_acoustic'
LOCAL_CKPT = '/content/DiffSinger/checkpoints/singer_v1_acoustic'
os.makedirs(DRIVE_CKPT, exist_ok=True)
if os.path.exists(LOCAL_CKPT) and not os.path.islink(LOCAL_CKPT):
    # Move any existing local checkpoints to Drive first
    !mv {LOCAL_CKPT}/* {DRIVE_CKPT}/ 2>/dev/null; rm -rf {LOCAL_CKPT}
if not os.path.exists(LOCAL_CKPT):
    os.symlink(DRIVE_CKPT, LOCAL_CKPT)

# Check if resuming from a previous run
import glob
existing = sorted(glob.glob(f'{DRIVE_CKPT}/model_ckpt_steps_*.ckpt'))
if existing:
    print(f'Resuming from: {os.path.basename(existing[-1])}')
    reset_flag = ''
else:
    print('Starting fresh (no previous checkpoints found)')
    reset_flag = '--reset'

!python scripts/train.py \
    --config configs/singer_v1_acoustic.yaml \
    --exp_name singer_v1_acoustic \
    $reset_flag

## 5. Monitor (optional)
Open TensorBoard in another cell while training runs.

In [ ]:
# Uncomment to launch TensorBoard
# %load_ext tensorboard
# %tensorboard --logdir /content/DiffSinger/checkpoints/singer_v1_acoustic/lightning_logs/

## 6. Download checkpoint

In [ ]:
import glob, os

ckpt_dir = '/content/drive/MyDrive/diffsinger_training/singer_v1_acoustic'
ckpts = sorted(glob.glob(f'{ckpt_dir}/model_ckpt_steps_*.ckpt'))
print('Available checkpoints:')
for c in ckpts:
    print(f'  {os.path.basename(c)} ({os.path.getsize(c) / 1e6:.1f} MB)')

In [ ]:
# Zip the checkpoint directory for download (already on Drive, but this makes a single file)
!cd /content/drive/MyDrive/diffsinger_training && zip -r /content/singer_v1_acoustic_ckpt.zip singer_v1_acoustic/ \
    -x 'singer_v1_acoustic/lightning_logs/*'

from google.colab import files
files.download('/content/singer_v1_acoustic_ckpt.zip')